# Weeks 3–4 Hands-on exercise

This hands-on exercise is a simplified replication of an example from Hernán and Robins, *Causal Inference: What If*, Chapter 12. The data come from the NHEFS (the National Health and Nutrition Examination Survey I Epidemiologic Follow-up Study): cigarette smokers who answered a first questionnaire in 1971 and were weighed again in 1982. Some of them quit smoking in between.

**Does quitting smoking cause weight gain?** Quitting was not randomized: people chose it, and the quitters may differ from the others in age, sex, how much they smoked, and so on. So we use the tools of Weeks 3 and 4: balance checks, nearest-neighbor matching, the propensity score, IPW, the regression estimator and AIPW.

Hernán and Robins estimate the average treatment effect (ATE) by IPW as **3.4 kg** (95% confidence interval 2.4 to 4.5). Our goal is to reproduce this number.

Save the notebook in your course folder, open it in Visual Studio Code, and choose the **ECON 481 (Python 3.12)** kernel. The data come with the `causaldata` package.

#### Q1. Write down the research question above in one sentence. What is the treatment, and what is the outcome?

*Your answer:*

#### Q2. Import the packages and load the data. We use `nhefs_complete`, the participants with complete data on the variables we need.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from causaldata import nhefs_complete, nhefs_codebook

df = nhefs_complete.load_pandas().data
print(df.shape)    # (1566, 67)

#### Q3. Read the codebook for the variables we use

The treatment is `qsmk`, the outcome is `wt82_71`, and the covariates, all measured in 1971, are `sex`, `race`, `age`, `education`, `smokeintensity`, `smokeyrs`, `exercise`, `active` and `wt71`. Print their descriptions with the codebook. What do the values of `education`, `exercise` and `active` mean?

In [ ]:
covariates = ["sex", "race", "age", "education", "smokeintensity", "smokeyrs", "exercise", "active", "wt71"]
codebook = nhefs_codebook.load_pandas().data
# Your code here

*Your answer:*

#### Q4. Data types

Look at `df[covariates].dtypes`. Some columns are stored as **categories** (text labels such as `"1"`), and `pandas` cannot average them. Convert `qsmk`, `sex`, `race`, `education`, `exercise` and `active` to integers with `.astype(int)`.

In [ ]:
print(df[covariates + ['qsmk']].dtypes)

# Your code here

#### Q5. The naive comparison

Split the data into quitters (`treated`, `qsmk == 1`) and non-quitters (`control`, `qsmk == 0`). Compute the mean weight change in each group and the difference, first **by hand** (sum divided by the number of people), then with `.mean()`.

In [ ]:
# Your code here

Is this difference the causal effect of quitting? Why or why not?

*Your answer:*

#### Q6. Are the groups comparable? A balance check (Week 3, Lab 2)

For each covariate, compute the mean among quitters, the mean among non-quitters, and the **standardized difference**

$$
\text{std. diff.} = \frac{\overline{X}_1 - \overline{X}_0}{\sqrt{(s_{X,1}^2 + s_{X,0}^2) / 2}}.
$$

Which covariates differ by more than 0.1 in absolute value?

In [ ]:
# Your code here

*Your answer:*

#### Q7. Nearest-neighbor matching (Week 3, Lab 5)

Match each quitter to the **closest** non-quitter, with replacement, using the Mahalanobis distance on the nine covariates, and estimate the effect on the quitters:

$$
\hat\tau_{\text{ATT}} = \frac{1}{N^1} \sum_{i:\, D_i = 1} \big(Y_i - Y_{j(i)}\big).
$$

**Your turn.** Read Lab 5.3 on the Week 3 page (<https://yoonismo.github.io/causal-computing/week3.html>) and adapt the function `nn_match` to these data. Then repeat with the Euclidean and the normalized Euclidean distance. Which distance would you trust, and why?

In [ ]:
X1 = treated[covariates].to_numpy(dtype=float)   # N1 x p covariates of the quitters
X0 = control[covariates].to_numpy(dtype=float)   # N0 x p covariates of the non-quitters
y1 = treated["wt82_71"].to_numpy()
y0 = control["wt82_71"].to_numpy()

# Your code here

*Your answer:*

#### Q8. The propensity score and overlap (Week 4, Lab 1)

Estimate $e(X_i) = P(D_i = 1 \mid X_i)$ with a logit. We use the same model as Hernán and Robins: the categorical covariates as dummies (`C(...)`) and squares of the four continuous ones.

Then plot the estimated scores of the quitters and the non-quitters (as in Week 4, Lab 1.3). Is there enough overlap for the ATE?

In [ ]:
ps_formula = ("qsmk ~ C(sex) + C(race) + age + I(age**2) + C(education)"
              " + smokeintensity + I(smokeintensity**2) + smokeyrs + I(smokeyrs**2)"
              " + C(exercise) + C(active) + wt71 + I(wt71**2)")

# Your code here

*Your answer:*

#### Q9. IPW (Week 4, Lab 2)

Compute the IPW estimate of the ATE in the Hájek form,

$$
\hat\tau_{\text{ATE}}^{\text{IPW}} = \frac{\sum_i D_i Y_i / \hat e(X_i)}{\sum_i D_i / \hat e(X_i)} - \frac{\sum_i (1 - D_i) Y_i / (1 - \hat e(X_i))}{\sum_i (1 - D_i) / (1 - \hat e(X_i))},
$$

and the IPW estimate of the ATT, with $\hat\omega_i = \hat e(X_i) / (1 - \hat e(X_i))$,

$$
\hat\tau_{\text{ATT}}^{\text{IPW}} = \frac{1}{N^1} \sum_{i:\, D_i = 1} Y_i - \frac{\sum_i (1 - D_i)\, \hat\omega_i Y_i}{\sum_i (1 - D_i)\, \hat\omega_i}.
$$

Compare the ATE with the 3.4 kg of Hernán and Robins, and the ATT with your matching estimate of Q7.

In [ ]:
# Your code here

*Your answer:*

#### Q10. The same number in one line, with a confidence interval

Hernán and Robins compute the IPW estimate as a **weighted regression** of the outcome on the treatment, with weight $1/\hat e(X_i)$ for the quitters and $1/(1 - \hat e(X_i))$ for the others, and a robust standard error. Run it with `smf.wls(..., weights=...)` and `.fit(cov_type="HC1")`. Do you get their estimate and their 95% interval (2.4 to 4.5)?

In [ ]:
# Your code here

*Your answer:*

#### Q11. The regression estimator (Week 4, Lab 3)

Fit the outcome model separately among quitters ($\hat\mu_1$) and non-quitters ($\hat\mu_0$), with the same terms as the propensity score, and average the difference over everyone:

$$
\hat\tau_{\text{ATE}}^{\text{reg}} = \frac{1}{n} \sum_{i=1}^{n} \big( \hat\mu_1(X_i) - \hat\mu_0(X_i) \big).
$$

In [ ]:
outcome_formula = ps_formula.replace("qsmk ~", "wt82_71 ~")
# Your code here

*Your answer:*

#### Q12. AIPW (Week 4, Lab 4)

Combine the two models:

$$
\hat\tau_{\text{ATE}}^{\text{AIPW}} = \frac{1}{n} \sum_{i=1}^{n} \left[ \hat\mu_1(X_i) - \hat\mu_0(X_i) + \frac{D_i \big(Y_i - \hat\mu_1(X_i)\big)}{\hat e(X_i)} - \frac{(1 - D_i)\big(Y_i - \hat\mu_0(X_i)\big)}{1 - \hat e(X_i)} \right].
$$

In [ ]:
# Your code here

*Your answer:*

#### Q13. Put it together

Make one table with the naive difference and all your estimates (matching, IPW, regression, AIPW). In a few sentences:

1. How much of the naive difference was due to the differences in the covariates?
2. All the methods rely on unconfoundedness. Name one variable that is **not** in the data and could affect both quitting and weight change. In which direction could it bias the estimates?

In [ ]:
# Your code here

*Your answer:*

**Reference.** Hernán, M. A., and Robins, J. M. *Causal Inference: What If*. Chapman & Hall/CRC. Chapter 12. Online version: <https://miguelhernan.org/whatifbook>